# 05. 로그 텍스트 처리


## Goal

작은 로그를 필터링·집계하고 결과를 검증합니다.


## Setup


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-05-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"새 임시 실습 디렉터리: {lab_dir}")


In [ ]:
%%bash
set -euo pipefail
cat > "$BASH_LAB_DIR/auth.log" <<'EOF'
2026-09-04T10:00:00Z alice SUCCESS 10.0.0.10
2026-09-04T10:01:00Z bob FAILED 10.0.0.20
2026-09-04T10:02:00Z alice FAILED 10.0.0.20
2026-09-04T10:03:00Z carol FAILED 10.0.0.30
2026-09-04T10:04:00Z bob FAILED 10.0.0.20
EOF


## Steps

### 1. 실패 이벤트만 선택


In [ ]:
%%bash
set -euo pipefail
grep ' FAILED ' "$BASH_LAB_DIR/auth.log" > "$BASH_LAB_DIR/failed.log"
cat "$BASH_LAB_DIR/failed.log"


### 2. 출발지 IP별 집계


In [ ]:
%%bash
set -euo pipefail
awk '$3 == "FAILED" { count[$4]++ } END { for (ip in count) print count[ip], ip }'   "$BASH_LAB_DIR/auth.log"   | sort -nr   | tee "$BASH_LAB_DIR/ip-summary.txt"


### 3. 결과에 대한 자동 점검


In [ ]:
%%bash
set -euo pipefail
failed_count=$(wc -l < "$BASH_LAB_DIR/failed.log" | tr -d ' ')
top_count=$(awk 'NR == 1 { print $1 }' "$BASH_LAB_DIR/ip-summary.txt")
[[ $failed_count -eq 4 ]]
[[ $top_count -eq 3 ]]
printf 'checks passed: failed=%s top_count=%s\n' "$failed_count" "$top_count"


## Checks

- 실패 이벤트가 정확히 4개인가?
- `10.0.0.20`이 3회로 가장 많은가?
- 공백 구분 로그가 아닌 CSV·JSON이라면 전용 파서나 Python을 선택해야 하는 이유를 설명할 수 있는가?


## Next Steps

운영체제의 상태를 읽기 전용으로 수집하고 안전 옵션을 적용합니다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
